In [ ]:
from pathlib import Path

import pandas as pd

from peakstay.api import fetch_all_shelters, load_raw, save_raw
from peakstay.cleaning import build_dataframe, clean_altitude, normalize_text

API CAI: 764 rifugi, 77 pagine da 10

In [ ]:
RAW_PATH = Path("../data/raw/shelters.json")

if RAW_PATH.exists():
    shelters = load_raw(RAW_PATH)
else:
    shelters = fetch_all_shelters()
    save_raw(shelters, RAW_PATH)

print(len(shelters))

In [ ]:
df = build_dataframe(shelters)
df = clean_altitude(df)
df = normalize_text(df)
print(df.shape)

In [ ]:
cols = ["region_geo", "ownerRegion_geo", "municipality_geo",
        "massif_geo", "locality_geo", "altitude_geo"]
print(df[cols].notna().sum())

print("comuni distinti:", df["municipality_geo"].nunique())
print("massicci distinti:", df["massif_geo"].nunique())

In [ ]:
per_mun = df["municipality_geo"].value_counts()
print((per_mun == 1).sum(), "comuni con un solo rifugio su", len(per_mun))
print(per_mun.head(10))

print(df["massif_geo"].value_counts().describe())

both = df.dropna(subset=["region_geo", "ownerRegion_geo"])
diff = both[both["region_geo"] != both["ownerRegion_geo"]]
print(len(diff))
print(diff[["title", "region_geo", "ownerRegion_geo"]].head(10))

service_cols = [c for c in df.columns if c.endswith("_service")]
print(len(service_cols))
print(df[service_cols].notna().sum().sort_values(ascending=False).head(15))

In [ ]:
service_cols = [c for c in df.columns if c.endswith("_service")]
print(len(service_cols))
print(pd.unique(df[service_cols].values.ravel()))